In [1]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape
from shapely import wkt
import warnings
# filter all warnings
warnings.filterwarnings("ignore")

In [2]:
with open('Data/Metro/metro_stops.json') as f:
    data = json.load(f)

df_metro_stops = []
for stop in data['features']:
    properties = stop.get('properties', {})
    nom_estacio = properties.get('NOM_ESTACIO')
    linia = properties.get('PICTO')
    id = properties.get('ID_ESTACIO')
    df_metro_stops.append({
            'id': id,
            'name': nom_estacio,
            'linia': linia,
            'stop_type': 'Metro',
            'geometry': shape(stop['geometry'])
        })


geo_df_metro_stops = gpd.GeoDataFrame(df_metro_stops, geometry='geometry')
excluded = ['FM','TM']
geo_df_metro_stops = geo_df_metro_stops[~geo_df_metro_stops['linia'].isin(excluded)]
geo_df_metro_stops

,id,name,linia,stop_type,geometry
0,111,Hospital de Bellvitge,L1,Metro,POINT (2.10724 41.34468)
1,112,Bellvitge,L1,Metro,POINT (2.11092 41.35097)
2,113,Av. Carrilet,L1,Metro,POINT (2.10265 41.35852)
3,114,Rambla Just Oliveras,L1,Metro,POINT (2.09975 41.36409)
4,115,Can Serra,L1,Metro,POINT (2.10275 41.36769)
...,...,...,...,...,...
133,959,Provençana,L10S,Metro,POINT (2.1241 41.36135)
134,1137,Casa de l'Aigua,L11,Metro,POINT (2.18484 41.45128)
135,1138,Torre Baró | Vallbona,L11,Metro,POINT (2.17988 41.4592)
136,1139,Ciutat Meridiana,L11,Metro,POINT (2.17465 41.46081)


# Solution nodes

Theya are the same as the original stops

fara falta un filtrat de parades les quals considerem possible soslucions

In [3]:
solution_nodes = geo_df_metro_stops.copy()
solution_nodes = solution_nodes.drop(columns=['linia'])
solution_nodes['id'] = 'SM' + '-' + solution_nodes['id'].astype(str)

# Metro Nodes

En fem un per cada Línia que tinguin

In [4]:
geo_df_metro_stops["linia"] = geo_df_metro_stops["linia"].str.findall(r"L\d+")
geo_df_metro_stops = geo_df_metro_stops.explode("linia", ignore_index=True)
geo_df_metro_stops['id'] = 'M' + '-' + geo_df_metro_stops['linia'] + '-' + geo_df_metro_stops['id'].astype(str)
geo_df_metro_stops


,id,name,linia,stop_type,geometry
0,M-L1-111,Hospital de Bellvitge,L1,Metro,POINT (2.10724 41.34468)
1,M-L1-112,Bellvitge,L1,Metro,POINT (2.11092 41.35097)
2,M-L1-113,Av. Carrilet,L1,Metro,POINT (2.10265 41.35852)
3,M-L1-114,Rambla Just Oliveras,L1,Metro,POINT (2.09975 41.36409)
4,M-L1-115,Can Serra,L1,Metro,POINT (2.10275 41.36769)
...,...,...,...,...,...
164,M-L10-959,Provençana,L10,Metro,POINT (2.1241 41.36135)
165,M-L11-1137,Casa de l'Aigua,L11,Metro,POINT (2.18484 41.45128)
166,M-L11-1138,Torre Baró | Vallbona,L11,Metro,POINT (2.17988 41.4592)
167,M-L11-1139,Ciutat Meridiana,L11,Metro,POINT (2.17465 41.46081)
